# Segmentación de cultivos con U-Net (ResNet-50) sobre PASTIS-R

Se entrena una U-Net de segmentación densa sobre las series Sentinel-2 de PASTIS-R y se evalúa su desempeño píxel a píxel. El encoder ResNet-50 viene preentrenado en ImageNet y se adapta a las diez bandas; como entrada se usa la mediana temporal de la serie y la salida es un mapa de clases a la resolución de la imagen.

Este cuaderno corre de forma independiente (en paralelo con el de AnySat) y deja sus artefactos en carpetas del Drive compartido para el reporte: la tabla de métricas, la figura de la matriz de confusión y el modelo entrenado.

## Datos y métricas

PASTIS-R entrega parches Sentinel-2 multitemporales de 128x128, que aquí se reescalan a 256. Las etiquetas tienen 20 clases: fondo, 18 tipos de cultivo y una clase void que se descarta en la pérdida y en las métricas. El split de entrenamiento y validación usa los folds oficiales del dataset, espacialmente disjuntos. Se reportan mIoU, F1-macro y exactitud a nivel de píxel en dos esquemas: las 18 clases planas y los 6 grupos agronómicos HCAT (cereales, oleaginosas, tubérculos, leguminosas, leñosos y otros), siendo este último el comparable con el baseline del avance anterior.

In [ ]:
# Setup del entorno. En Colab se monta Drive (donde vive el dataset) y se
# instalan las dependencias que no vienen por defecto; en local no hace falta.
import os, sys, subprocess
from pathlib import Path

_IN_COLAB = False
shared_folder_path = ''
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shared_folder_path = '/content/drive/MyDrive/Integrador/'
    _IN_COLAB = True
except ImportError:
    pass

# En Colab el repo no esta presente: se clona una vez en /content/agrosat-copilot.
if _IN_COLAB:
    from getpass import getpass
    _repo_dir = '/content/agrosat-copilot'
    _branch = 'users/abocanegra/unet-anysat'
    _repo = 'github.com/ArthurZizumbo/agrosat-copilot.git'
    if not Path(_repo_dir, 'pyproject.toml').is_file():
        _rc = os.system(f'git clone --branch {_branch} --depth 1 https://{_repo} {_repo_dir}')
        if _rc != 0:  # repo privado: pide token (no se guarda en el notebook)
            _tok = getpass('GitHub token (repo privado): ')
            os.system(f'git clone --branch {_branch} --depth 1 https://{_tok}@{_repo} {_repo_dir}')

# El codigo no vive en Drive: se localiza el repo por su pyproject.toml.
_search = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if _IN_COLAB:
    _search = [Path('/content/agrosat-copilot'), *_search]
for _cand in _search:
    if (_cand / 'pyproject.toml').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        os.chdir(_cand)
        break
else:
    raise RuntimeError('No se encontro el repo agrosat-copilot (pyproject.toml). '
                       'Clonalo en /content/agrosat-copilot o sincronizalo desde VS Code.')

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'segmentation-models-pytorch', 'structlog', 'typer', 'polars', 'mlflow'], check=False)

print('repo:', Path.cwd(), '| colab:', _IN_COLAB, '| drive:', shared_folder_path or '(local)')

In [ ]:
# Configuracion de la corrida.
import torch

MODEL = 'unet'
# El dataset vive en Drive; en local se usa la copia del repo.
PASTIS_ROOT = Path((shared_folder_path + 'data/PASTIS-R') if shared_folder_path
                   else 'data/PASTIS-R')
# Carpetas de artefactos en Drive (claras para citarlas en el reporte):
#   reports/segmentation/metrics      -> parquet de metricas por modelo
#   reports/segmentation/figures      -> PNG de la matriz de confusion
#   reports/segmentation/checkpoints  -> modelo final + checkpoint reanudable
SEG_DIR = Path((shared_folder_path if shared_folder_path else '') + 'reports/segmentation')
METRICS_DIR = SEG_DIR / 'metrics'
FIGURES_DIR = SEG_DIR / 'figures'
CHECKPOINT_DIR = SEG_DIR / 'checkpoints_fast'
for _d in (METRICS_DIR, FIGURES_DIR, CHECKPOINT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
COMPARISON_PATH = METRICS_DIR / f'model_comparison_avance4_{MODEL}_fast.parquet'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET_SIZE = 256
SUBSET = 0            # 0 = todos; reducir (p.ej. 600) si la sesion es corta
EPOCHS = 30
BATCH = 8
MLFLOW_URI = 'file:./mlruns'
# Por defecto se lee directo de Drive (sin copiar). Si vas a entrenar muchas epocas y
# preferis acelerar, pone COPY_TO_LOCAL=True (copia una vez al disco efimero).
COPY_TO_LOCAL = False
# Leyendo de Drive, num_workers=0 va mas rapido (el FUSE de Drive penaliza la
# concurrencia). Con el dataset copiado a local conviene subirlo a 2-4.
NUM_WORKERS = 0

print('modelo:', MODEL, '| device:', DEVICE, '| batch:', BATCH)
print('PASTIS_ROOT:', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())
print('artefactos en:', SEG_DIR)

## Lectura del dataset

Por defecto el dataset se lee directo desde Drive, sin copiar nada: así se evita la espera inicial y no se pierde trabajo si la sesión se reinicia. El loader abre cada parche con un solo acceso a disco (no relee el archivo de metadatos en cada paso) y el DataLoader usa varios procesos en paralelo. Si preferís acelerar, poné `COPY_TO_LOCAL = True` en la celda anterior para copiar una vez al disco local de la sesión.

In [ ]:
# Copia del dataset de Drive al disco local, con barra de progreso.
import shutil, time

def copy_pastis_to_local(src_root, dst_root,
                         subdirs=('DATA_S2', 'ANNOTATIONS'),
                         files=('metadata.geojson', 'NORM_S2_patch.json')):
    src_root, dst_root = Path(src_root), Path(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)
    todo = []
    for sub in subdirs:
        for f in sorted((src_root / sub).glob('*')):
            if f.is_file():
                todo.append((f, dst_root / sub / f.name))
    for fname in files:
        sp = src_root / fname
        if sp.is_file():
            todo.append((sp, dst_root / fname))
    if not todo:
        raise FileNotFoundError(f'No se hallaron DATA_S2/ANNOTATIONS en {src_root}')
    total_bytes = sum(s.stat().st_size for s, _ in todo)
    try:
        from tqdm.auto import tqdm
        bar = tqdm(total=total_bytes, unit='B', unit_scale=True, desc='Copiando PASTIS')
    except Exception:
        bar = None
    t0 = time.time()
    for i, (src, dst) in enumerate(todo, 1):
        dst.parent.mkdir(parents=True, exist_ok=True)
        # Salta el archivo si ya esta copiado con el mismo tamano.
        if not (dst.exists() and dst.stat().st_size == src.stat().st_size):
            shutil.copy2(src, dst)
        if bar is not None:
            bar.update(src.stat().st_size)
        elif i % 200 == 0:
            print(f'  {i}/{len(todo)} archivos...')
    if bar is not None:
        bar.close()
    print(f'Listo: {len(todo)} archivos ({total_bytes / 1e9:.1f} GB) en {time.time() - t0:.0f}s -> {dst_root}')
    return dst_root

if _IN_COLAB and COPY_TO_LOCAL:
    PASTIS_ROOT = copy_pastis_to_local(PASTIS_ROOT, '/content/PASTIS-R')
    print('PASTIS_ROOT (local):', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())
else:
    print('Lectura directa desde:', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())

In [ ]:
# Split en los folds oficiales de PASTIS (espacialmente disjuntos).
from ml.ingest.pastis_dataset import pastis_fold_split

split = pastis_fold_split(PASTIS_ROOT, train_folds=(1, 2, 3), val_folds=(4,), test_folds=(5,))
print({k: len(v) for k, v in split.items()})

## Entrenamiento

El entrenamiento guarda un checkpoint por época en Drive; si la sesión se reinicia, al volver a ejecutar esta celda se reanuda desde la última época completada en vez de empezar de cero.

In [ ]:
# Entrenamiento de la U-Net.
from ml.train.train_segmentation import run_training

result = run_training(
    model=MODEL, epochs=EPOCHS, batch_size=BATCH, target_size=TARGET_SIZE,
    subset=SUBSET, device=DEVICE, root=PASTIS_ROOT, mlflow_uri=MLFLOW_URI,
    comparison_path=COMPARISON_PATH, num_workers=NUM_WORKERS, output_dir=CHECKPOINT_DIR,
)
result

## Métricas

Tabla de métricas de este modelo sobre el fold de validación, en los dos esquemas (18 clases y 6 grupos HCAT). Se guarda en `reports/segmentation/metrics/`; el notebook integrador la une con la del otro modelo para la comparativa final. Las columnas con sufijo `grouped` corresponden a los 6 grupos (el fondo no entra en esas métricas).

In [ ]:
import polars as pl

table = pl.read_parquet(COMPARISON_PATH)
cols = ['model', 'miou_grouped', 'f1_macro_grouped', 'pixel_accuracy_grouped',
        'miou', 'f1_macro', 'pixel_accuracy', 'train_time_s', 'epochs']
table.select([c for c in cols if c in table.columns])

## Matriz de confusión

Recall por clase a nivel de píxel sobre el fold de validación, sin contar la clase void. La figura se guarda en `reports/segmentation/figures/` para el reporte.

In [ ]:
# Matriz de confusion a nivel de pixel; se guarda como PNG en Drive.
import torch
from torch.utils.data import DataLoader
from ml.ingest.pastis_dataset import PASTISDataset, load_norm_stats, PASTIS_IGNORE_INDEX
from ml.ingest.pastis_loader import PASTIS_CLASS_MAP
from ml.eval.dense_metrics import dense_confusion_figure
from ml.models.segmentation import build_unet

def confusion_figure(model_name, reduction, build_fn, ckpt, max_patches=40):
    norm = load_norm_stats(PASTIS_ROOT, folds=(1, 2, 3))
    val_ids = split['val'][:max_patches]
    ds = PASTISDataset(val_ids, root=PASTIS_ROOT, target_size=TARGET_SIZE,
                       temporal_reduction=reduction, norm=norm)
    loader = DataLoader(ds, batch_size=2)
    model = build_fn().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    preds, tgts = [], []
    with torch.no_grad():
        for b in loader:
            img = b['image'].to(DEVICE)
            out = model(img) if model_name == 'unet' else model(img, b['dates'].to(DEVICE))
            preds.append(out.argmax(1).cpu().reshape(-1))
            tgts.append(b['semantic'].reshape(-1))
    return dense_confusion_figure(torch.cat(preds), torch.cat(tgts),
                                  class_names=PASTIS_CLASS_MAP, ignore_index=PASTIS_IGNORE_INDEX)

fig = confusion_figure(MODEL, 'median', lambda: build_unet(20, encoder_weights=None),
                       result['checkpoint_path'])
_fig_path = FIGURES_DIR / f'confusion_{MODEL}_fast.png'
fig.savefig(_fig_path, bbox_inches='tight', dpi=120)
print('Figura guardada en:', _fig_path)
fig

## Conclusiones

Las métricas y la matriz de confusión quedan guardadas en `reports/segmentation/` (carpetas `metrics/` y `figures/`) y el modelo entrenado en `checkpoints/`. El notebook integrador `Avance4.Equipo17` reúne este modelo con el otro para la comparativa final, elige el de mejor desempeño y, si vale la pena, afina sus hiperparámetros con una búsqueda más fina como la del bloque siguiente.

In [ ]:
# Busqueda de hiperparametros con Optuna (opcional, si este modelo entra al top).
#
# import optuna
# def objective(trial):
#     lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
#     wd = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
#     res = run_training(model=MODEL, epochs=15, batch_size=BATCH, lr=lr, weight_decay=wd,
#                        target_size=TARGET_SIZE, subset=SUBSET, device=DEVICE,
#                        root=PASTIS_ROOT, mlflow_uri=MLFLOW_URI, resume=False)
#     return res['miou_grouped']
# study = optuna.create_study(direction='maximize', study_name=f'tune-{MODEL}')
# study.optimize(objective, n_trials=30)
# study.best_params